In [26]:
import pandas as pd
import numpy as np

# --- 1. Load datasets ---
customer_df = pd.read_csv("Customer_Feedback_Data.csv")
product_df = pd.read_csv("Product_Offering_Data.csv")
transactions_df = pd.read_csv("Transaction_Data.csv")

# --- 2. Handle missing values ---

# Customer Feedback: fill missing Satisfaction_Score with mean
customer_df['Satisfaction_Score'] = customer_df['Satisfaction_Score'].fillna(
    customer_df['Satisfaction_Score'].mean()
)

# Product Offering: drop Target_Age_Group (all values missing)
if 'Target_Age_Group' in product_df.columns:
    product_df = product_df.drop(columns=['Target_Age_Group'])

# Transactions: fill missing Transaction_Amount with median
transactions_df['Transaction_Amount'] = transactions_df['Transaction_Amount'].fillna(
    transactions_df['Transaction_Amount'].median()
)

# --- 3. Convert data types ---

# Transactions: convert Transaction_Date to datetime
transactions_df['Transaction_Date'] = pd.to_datetime(transactions_df['Transaction_Date'])

# Optional: check dtypes after conversion
print(customer_df.dtypes)
print(product_df.dtypes)
print(transactions_df.dtypes)

# --- 4. Check for duplicates ---
customer_df = customer_df.drop_duplicates()
product_df = product_df.drop_duplicates()
transactions_df = transactions_df.drop_duplicates()

# --- 5. Reset index after cleaning ---
customer_df = customer_df.reset_index(drop=True)
product_df = product_df.reset_index(drop=True)
transactions_df = transactions_df.reset_index(drop=True)

# --- 6. Quick check of missing values after cleaning ---
print("Customer Feedback missing values:\n", customer_df.isnull().sum())
print("Product Offering missing values:\n", product_df.isnull().sum())
print("Transactions missing values:\n", transactions_df.isnull().sum())

Customer_ID                  int64
Satisfaction_Score         float64
Feedback_Comments           object
Likelihood_to_Recommend      int64
dtype: object
Product_ID              int64
Product_Name           object
Product_Type           object
Risk_Level             object
Target_Income_Group    object
dtype: object
Transaction_ID                 int64
Customer_ID                    int64
Transaction_Date      datetime64[ns]
Transaction_Amount           float64
Transaction_Type              object
dtype: object
Customer Feedback missing values:
 Customer_ID                0
Satisfaction_Score         0
Feedback_Comments          0
Likelihood_to_Recommend    0
dtype: int64
Product Offering missing values:
 Product_ID             0
Product_Name           0
Product_Type           0
Risk_Level             0
Target_Income_Group    0
dtype: int64
Transactions missing values:
 Transaction_ID        0
Customer_ID           0
Transaction_Date      0
Transaction_Amount    0
Transaction_Type     

In [27]:
# --- Map Risk_Level to numeric for aggregation ---
risk_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
product_df['Risk_Level_Num'] = product_df['Risk_Level'].map(risk_mapping)

# --- Merge transactions with product info ---
transactions_merged = transactions_df.merge(
    product_df[['Product_ID', 'Product_Type', 'Risk_Level_Num']],
    left_on='Transaction_Type',  # or Product_ID if available
    right_on='Product_Type',     # adjust depending on dataset
    how='left'
)

# --- Customer-level aggregation ---
customer_features = transactions_merged.groupby('Customer_ID').agg(
    Total_Transactions=('Transaction_ID', 'count'),
    Total_Amount=('Transaction_Amount', 'sum'),
    Avg_Amount=('Transaction_Amount', 'mean'),
    Avg_Risk_Level=('Risk_Level_Num', 'mean'),
    Most_Frequent_Product_Type=('Product_Type', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    Last_Transaction=('Transaction_Date', 'max')
).reset_index()

# --- Add feedback features ---
customer_features = customer_features.merge(
    customer_df[['Customer_ID', 'Satisfaction_Score', 'Likelihood_to_Recommend']],
    on='Customer_ID',
    how='left'
)

# --- Calculate days since last transaction ---
customer_features['Days_Since_Last_Transaction'] = (
    pd.Timestamp.today() - customer_features['Last_Transaction']
).dt.days

# Quick look
customer_features.head()

,Customer_ID,Total_Transactions,Total_Amount,Avg_Amount,Avg_Risk_Level,Most_Frequent_Product_Type,Last_Transaction,Satisfaction_Score,Likelihood_to_Recommend,Days_Since_Last_Transaction
0,1,7,18114.0,2587.714286,3.0,Investment,2023-07-02 03:00:00,10.0,9,951
1,1,7,18114.0,2587.714286,3.0,Investment,2023-07-02 03:00:00,7.0,9,951
2,2,2,4907.0,2453.500000,NaN,NaN,2023-05-21 23:00:00,3.0,3,992
3,2,2,4907.0,2453.500000,NaN,NaN,2023-05-21 23:00:00,7.0,5,992
4,2,2,4907.0,2453.500000,NaN,NaN,2023-05-21 23:00:00,3.0,7,992


In [28]:
import pandas as pd
import numpy as np

# --- Map Risk_Level to numeric for aggregation ---
risk_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
product_df['Risk_Level_Num'] = product_df['Risk_Level'].map(risk_mapping)

# --- Merge transactions with product info based on Transaction_Type matching Product_Type ---
transactions_merged = transactions_df.merge(
    product_df[['Product_Type', 'Risk_Level_Num']],
    left_on='Transaction_Type',
    right_on='Product_Type',
    how='left'
)

# --- Customer-level aggregation ---
customer_features = transactions_merged.groupby('Customer_ID').agg(
    Total_Transactions=('Transaction_ID', 'count'),
    Total_Amount=('Transaction_Amount', 'sum'),
    Avg_Amount=('Transaction_Amount', 'mean'),
    Avg_Risk_Level=('Risk_Level_Num', 'mean'),
    Most_Frequent_Product_Type=('Transaction_Type', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    Last_Transaction=('Transaction_Date', 'max')
).reset_index()

# --- Add feedback features ---
customer_features = customer_features.merge(
    customer_df[['Customer_ID', 'Satisfaction_Score', 'Likelihood_to_Recommend']],
    on='Customer_ID',
    how='left'
)

# --- Calculate days since last transaction ---
customer_features['Days_Since_Last_Transaction'] = (
    pd.Timestamp.today() - customer_features['Last_Transaction']
).dt.days

# --- Optional: drop Last_Transaction if not needed ---
customer_features.drop(columns=['Last_Transaction'], inplace=True)

# --- Preview the engineered dataset ---
customer_features.head()

,Customer_ID,Total_Transactions,Total_Amount,Avg_Amount,Avg_Risk_Level,Most_Frequent_Product_Type,Satisfaction_Score,Likelihood_to_Recommend,Days_Since_Last_Transaction
0,1,7,18114.0,2587.714286,3.0,Bill Payment,10.0,9,951
1,1,7,18114.0,2587.714286,3.0,Bill Payment,7.0,9,951
2,2,2,4907.0,2453.500000,NaN,Bill Payment,3.0,3,992
3,2,2,4907.0,2453.500000,NaN,Bill Payment,7.0,5,992
4,2,2,4907.0,2453.500000,NaN,Bill Payment,3.0,7,992


In [31]:
import pandas as pd
import numpy as np

# Load datasets
customer_df = pd.read_csv('Customer_Feedback_Data.csv')
product_df = pd.read_csv('Product_Offering_Data.csv')
transactions_df = pd.read_csv('Transaction_Data.csv')

# -----------------------
# 1. Clean Customer Feedback
# -----------------------
# Fill missing Satisfaction_Score with mean
customer_df['Satisfaction_Score'] = customer_df['Satisfaction_Score'].fillna(
    customer_df['Satisfaction_Score'].mean()
)

# -----------------------
# 2. Clean Product Offering
# -----------------------
# Drop Target_Age_Group since it's completely empty
if 'Target_Age_Group' in product_df.columns:
    product_df = product_df.drop(columns=['Target_Age_Group'])

# -----------------------
# 3. Clean Transactions
# -----------------------
# Convert Transaction_Date to datetime
transactions_df['Transaction_Date'] = pd.to_datetime(transactions_df['Transaction_Date'])

# Fill missing Transaction_Amount with median
transactions_df['Transaction_Amount'] = transactions_df['Transaction_Amount'].fillna(
    transactions_df['Transaction_Amount'].median()
)

# -----------------------
# 4. Check results
# -----------------------
print("Customer Feedback dtypes:\n", customer_df.dtypes)
print("\nProduct Offering dtypes:\n", product_df.dtypes)
print("\nTransactions dtypes:\n", transactions_df.dtypes)

print("\nMissing values - Customer Feedback:\n", customer_df.isna().sum())
print("\nMissing values - Product Offering:\n", product_df.isna().sum())
print("\nMissing values - Transactions:\n", transactions_df.isna().sum())

# -----------------------
# 5. Save cleaned datasets
# -----------------------
customer_df.to_csv('Customer_Feedback_Cleaned.csv', index=False)
product_df.to_csv('Product_Offering_Cleaned.csv', index=False)
transactions_df.to_csv('Transaction_Data_Cleaned.csv', index=False)

print("\nCleaned files saved:")
print("- Customer_Feedback_Cleaned.csv")
print("- Product_Offering_Cleaned.csv")
print("- Transaction_Data_Cleaned.csv")

Customer Feedback dtypes:
 Customer_ID                  int64
Satisfaction_Score         float64
Feedback_Comments           object
Likelihood_to_Recommend      int64
dtype: object

Product Offering dtypes:
 Product_ID              int64
Product_Name           object
Product_Type           object
Risk_Level             object
Target_Income_Group    object
dtype: object

Transactions dtypes:
 Transaction_ID                 int64
Customer_ID                    int64
Transaction_Date      datetime64[ns]
Transaction_Amount           float64
Transaction_Type              object
dtype: object

Missing values - Customer Feedback:
 Customer_ID                0
Satisfaction_Score         0
Feedback_Comments          0
Likelihood_to_Recommend    0
dtype: int64

Missing values - Product Offering:
 Product_ID             0
Product_Name           0
Product_Type           0
Risk_Level             0
Target_Income_Group    0
dtype: int64

Missing values - Transactions:
 Transaction_ID        0
Custome

In [32]:
import pandas as pd
from datetime import datetime

# Load CLEANED datasets
customer_df = pd.read_csv('Customer_Feedback_Cleaned.csv')
product_df = pd.read_csv('Product_Offering_Cleaned.csv')  # not used yet, but kept for project
transactions_df = pd.read_csv('Transaction_Data_Cleaned.csv')

# Make sure date is datetime
transactions_df['Transaction_Date'] = pd.to_datetime(transactions_df['Transaction_Date'])

# -----------------------
# 1. Aggregate transactions per customer
# -----------------------
txn_features = transactions_df.groupby('Customer_ID').agg(
    Total_Transactions=('Transaction_ID', 'count'),
    Total_Amount=('Transaction_Amount', 'sum'),
    Avg_Transaction_Amount=('Transaction_Amount', 'mean'),
    Max_Transaction_Amount=('Transaction_Amount', 'max'),
    Last_Transaction_Date=('Transaction_Date', 'max')
).reset_index()

# -----------------------
# 2. Create Recency Feature
# -----------------------
today = pd.to_datetime("today")
txn_features['Days_Since_Last_Transaction'] = (today - txn_features['Last_Transaction_Date']).dt.days

# Drop helper column
txn_features = txn_features.drop(columns=['Last_Transaction_Date'])

# -----------------------
# 3. Merge with customer feedback
# -----------------------
customer_features = pd.merge(
    txn_features,
    customer_df[['Customer_ID', 'Satisfaction_Score', 'Likelihood_to_Recommend']],
    on='Customer_ID',
    how='left'
)

# -----------------------
# 4. Check result
# -----------------------
print(customer_features.head())
print("\nMissing values:\n", customer_features.isna().sum())
print("\nData types:\n", customer_features.dtypes)

# -----------------------
# 5. Save feature-engineered dataset
# -----------------------
customer_features.to_csv('FinMark_Customer_Features.csv', index=False)

print("\nSaved: FinMark_Customer_Features.csv")

   Customer_ID  Total_Transactions  Total_Amount  Avg_Transaction_Amount  \
0            1                   6       16836.0                  2806.0   
1            1                   6       16836.0                  2806.0   
2            2                   2        4907.0                  2453.5   
3            2                   2        4907.0                  2453.5   
4            2                   2        4907.0                  2453.5   

   Max_Transaction_Amount  Days_Since_Last_Transaction  Satisfaction_Score  \
0                  4993.0                          951                10.0   
1                  4993.0                          951                 7.0   
2                  2850.0                          992                 3.0   
3                  2850.0                          992                 7.0   
4                  2850.0                          992                 3.0   

   Likelihood_to_Recommend  
0                        9  
1               